In [1]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils(
    "Structured Streaming with Files",
    master_url="spark://spark-master:7077"
)

su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 19:53:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
import subprocess
result = subprocess.run(["find", "/opt/spark/work-dir", "-name", "*.py"], capture_output=True, text=True)
print(result.stdout)

/opt/spark/work-dir/src/sebastianM/spark_utils.py
/opt/spark/work-dir/src/session5/example2.py
/opt/spark/work-dir/src/emr_code/emr-example.py
/opt/spark/work-dir/src/pcamarillor/spark_utils.py
/opt/spark/work-dir/src/lecture_05_classes.py
/opt/spark/work-dir/src/session7/image_utils.py
/opt/spark/work-dir/src/utils.py



In [2]:
import sys
sys.path.insert(0, "/opt/spark/work-dir/lib")

from generate_logs import generate_log_lines, make_filename
import os

INPUT_PATH = "/opt/spark/work-dir/data/streaming/logs/"
os.makedirs(INPUT_PATH, exist_ok=True)

# Write 3 files — Spark will pick them up one per micro-batch
for i in range(1, 4):
    content  = generate_log_lines(n_lines=50)
    filename = make_filename(i)
    path     = os.path.join(INPUT_PATH, filename)
    with open(path, "w") as f:
        f.write(content)
    print(f"[{i}/3] Written: {path}")

ModuleNotFoundError: No module named 'generate_logs'

In [ ]:
import pyspark.sql.functions as F
from pathlib import Path
import shutil

logs_schema = SparkUtils.generate_schema([("raw_line", "string")])

logs_df = (
    su.spark.readStream
    .format("text")
    .option("maxFilesPerTrigger", 1)   # one file = one micro-batch
    .schema(logs_schema)
    .load(INPUT_PATH)
)